In [1]:
import sys
from pathlib import Path

import UnitMatchPy.Bayes_fun as bf
import UnitMatchPy.utils as util
import UnitMatchPy.Overlord as ov
import numpy as np
import matplotlib.pyplot as plt
import UnitMatchPy.Save_utils as su
import UnitMatchPy.GUI as gui
import UnitMatchPy.AssignUniqueID as aid
import pdb

In [2]:
#get default parameters, can add your own before or after!
param = util.get_default_param()

#Give the paths to the KS directories for each session
#If you don't have a dir with channel_positions.npy etc look at the detailed example for supplying paths separately 
KSdirs = [
    r"W:\branco\Laurence\JAL006\JAL006_shelter_barrier_flip_3_2024_03_21T11_20_34\barrier_flip3_g0\barrier_flip3_g0_imec0\SI_KS_output\sorter_output",
    r"W:\branco\Laurence\JAL006\JAL006_shelter_barrier_flip_5_2024_03_25T11_05_33\barrier_flip5_g0\barrier_flip5_g0_imec0\SI_KS_output\sorter_output"
]
#mport pdb; pdb.pm()
WavePaths, UnitLabelPaths, ChannelPos = util.paths_fromKS(KSdirs)

In [4]:
#import pdb; pdb.pm()

#read in data and select the good units and exact metadata
waveform, SessionID, SessionSwitch, WithinSession, GoodUnits, param = util.load_good_waveforms(WavePaths, UnitLabelPaths, param, GoodUnitsOnly = True) 

#param['PeakLoc'] = #may need to set as a value if the peak locatioon is NOT ~ half the spike width

# create clusInfo, contains all unit id/session related info
ClusInfo = {'GoodUnits' : GoodUnits, 'SessionSwitch' : SessionSwitch, 'SessionID' : SessionID, 
            'OriginalID' : np.concatenate(GoodUnits)}

import pdb; pdb.set_trace()


#Extract parameters from waveform
print(f"ChannelPos shape: {ChannelPos[0].shape}")
ExtractedWaveProperties = ov.extract_parameters(waveform, ChannelPos, ClusInfo, param)

#Extract metric scores
TotalScore, CandidatePairs, Scores2Include, Predictors  = ov.extract_metric_scores(ExtractedWaveProperties, SessionSwitch, WithinSession, param, niter  = 2)

#Probability analysis
priorMatch = 1 - (param['nExpectedMatches'] / param['nUnits']**2 ) # fredom of choose in prior prob?
Priors = np.array((priorMatch, 1-priorMatch))

labels = CandidatePairs.astype(int)
Cond = np.unique(labels)
ScoreVector = param['ScoreVector']
ParameterKernels = np.full((len(ScoreVector), len(Scores2Include), len(Cond)), np.nan)

ParameterKernels = bf.get_ParameterKernels(Scores2Include, labels, Cond, param, addone = 1)

Probability = bf.apply_naive_bayes(ParameterKernels, Priors, Predictors, param, Cond)

Output = Probability[:,1].reshape(param['nUnits'],param['nUnits'])

--Return--
None
> c:\users\laurence\appdata\local\temp\ipykernel_50524\2780240047.py(12)<module>()

array([121], dtype=object)
array([[2],
       [3],
       [4],
       [5],
       [7],
       [13],
       [15],
       [16],
       [17],
       [21],
       [24],
       [27],
       [29],
       [30],
       [33],
       [35],
       [36],
       [37],
       [39],
       [40],
       [41],
       [42],
       [43],
       [46],
       [48],
       [49],
       [53],
       [56],
       [58],
       [59],
       [60],
       [61],
       [62],
       [63],
       [65],
       [66],
       [76],
       [81],
       [82],
       [83],
       [86],
       [87],
       [90],
       [91],
       [97],
       [101],
       [102],
       [107],
       [108],
       [110],
       [111],
       [112],
       [113],
       [115],
       [118],
       [119],
       [121],
       [122],
       [123],
       [128],
       [129],
       [130],
       [136],
       [137],
       [138],
       [141],

In [ ]:
util.evaluate_output(Output, param, WithinSession, SessionSwitch, MatchThreshold = 0.75)

MatchThreshold = param['MatchThreshold']
#MatchThreshold = try different values here!

OutputThreshold = np.zeros_like(Output)
OutputThreshold[Output > MatchThreshold] = 1

plt.imshow(OutputThreshold, cmap = 'Greys')

In [ ]:
Amplitude = ExtractedWaveProperties['Amplitude']
SpatialDecay = ExtractedWaveProperties['SpatialDecay']
AvgCentroid = ExtractedWaveProperties['AvgCentroid']
AvgWaveform = ExtractedWaveProperties['AvgWaveform']
AvgWaveformPerTP = ExtractedWaveProperties['AvgWaveformPerTP']
WaveIdx = ExtractedWaveProperties['WaveIdx']
MaxSite = ExtractedWaveProperties['MaxSite']
MaxSiteMean = ExtractedWaveProperties['MaxSiteMean']
gui.process_info_for_GUI(Output, MatchThreshold, Scores2Include, TotalScore, Amplitude, SpatialDecay,
                         AvgCentroid, AvgWaveform, AvgWaveformPerTP, WaveIdx, MaxSite, MaxSiteMean, 
                         waveform, WithinSession, ChannelPos, ClusInfo, param)

In [ ]:
IsMatch, NotMatch, MatchesGUI = gui.run_GUI()

In [ ]:
Matches = np.argwhere(MatchThreshold == 1)
UIDs = aid.AssignUID(Output, param, ClusInfo)

SaveDir = r'Path/to/save/directory'
su.save_to_output(SaveDir, Scores2Include, Matches, Output, AvgCentroid, AvgWaveform, AvgWaveformPerTP, MaxSite,
                   TotalScore, OutputThreshold, ClusInfo, param, UIDs = UIDs, MatchesCurated = None, SaveMatchTable = True)